# Pythia 160m: Accessibility Domain Knowledge

12 layers | 12 heads | d_model=768 | d_mlp=3072 | parallel attn+MLP

In [1]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [2]:
model_name = "pythia-160m"
model, info = load_model(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-160m into HookedTransformer
Loaded pythia-160m on mps
  12 layers | 12 heads | d_model=768 | d_mlp=3072 | parallel attn+MLP


In [3]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: pythia-160m
Prompt: "The W in WCAG stands for"
Target: " Web" (token 7066)
Final prediction: " the"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         example             36075          0.000000    
1         instance            38102          0.000000    
2         raised              7797           0.000000    
3         ‘                   13582          0.000000    
4         “                   9630           0.000000    
5         advanced            7735           0.000000    
6         advanced            9158           0.000000    
7         advanced            1197           0.000000    
8         W                   17             0.000004    
9         World               3              0.080258    
10        W                   7              0.008760    
11        the                 5              0.036509    
Model: pythia-160m
Target: " Web"

La

In [4]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [6]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/pythia/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/pythia/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/pythia/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/pythia/{model_name}/figures")

Saved: ../../results/pythia/pythia-160m/figures/wcag-logit-lens.png
Saved: ../../results/pythia/pythia-160m/figures/wcag-decomposition.png
Saved: ../../results/pythia/pythia-160m/figures/wcag-head-heatmap.png
Saved: ../../results/pythia/pythia-160m/figures/aria-logit-lens.png
Saved: ../../results/pythia/pythia-160m/figures/aria-decomposition.png
Saved: ../../results/pythia/pythia-160m/figures/aria-head-heatmap.png
Saved: ../../results/pythia/pythia-160m/figures/alt-logit-lens.png
Saved: ../../results/pythia/pythia-160m/figures/alt-decomposition.png
Saved: ../../results/pythia/pythia-160m/figures/alt-head-heatmap.png
Saved: ../../results/pythia/pythia-160m/figures/HTML-logit-lens.png
Saved: ../../results/pythia/pythia-160m/figures/HTML-decomposition.png
Saved: ../../results/pythia/pythia-160m/figures/HTML-head-heatmap.png
Saved: ../../results/pythia/pythia-160m/figures/screenReader-logit-lens.png
Saved: ../../results/pythia/pythia-160m/figures/screenReader-decomposition.png
Saved: ../..

In [7]:
unload(model)

Model unloaded, memory cleared
